[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 01](README.md)

# Jerarquía de memoria y Roofline

**Tema:** 01 · **Sesiones:** 5, 6 · **Edición:** 1.0.2026

**Pregunta guía:** ¿La ejecución está limitada por cómputo, ancho de banda, latencia o localidad?


## Resultados de aprendizaje

- Calcular intensidad aritmética.
- Aplicar el límite Roofline sin confundirlo con una medición.
- Reconocer localidad, NUMA y false sharing como causas observables.


## Modelo conceptual

Roofline acota rendimiento por min(pico, ancho de banda × intensidad).

Una línea de caché compartida por escrituras de varios núcleos puede invalidarse aunque las variables sean distintas.

En NUMA, ubicación de memoria, first-touch y afinidad forman parte del experimento.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "01"
NOTEBOOK = "01_fundamentos/03_memoria_roofline.ipynb"
assert (ROOT / "curso" / "notebooks" / "01_fundamentos" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Límite Roofline

Se calcula el techo para varias intensidades con unidades coherentes.


In [ ]:
peak_gflops = 800.0
bandwidth_gbs = 120.0
intensities = (0.125, 0.5, 1, 2, 4, 8, 16)
ridge = peak_gflops / bandwidth_gbs
assert abs(ridge - 20/3) < 1e-12
print(f"punto de transición = {ridge:.2f} FLOP/byte")
for intensity in intensities:
    ceiling = min(peak_gflops, bandwidth_gbs * intensity)
    assert 0 < ceiling <= peak_gflops
    regime = "memoria" if bandwidth_gbs * intensity < peak_gflops else "cómputo"
    print(f"I={intensity:6.3f} techo={ceiling:7.1f} GFLOP/s régimen={regime}")


**Interpretación.** El techo no es rendimiento obtenido: se compara con mediciones de la misma precisión y operación.


## Líneas de caché

Se observa cuándo contadores adyacentes comparten una línea de 64 bytes.


In [ ]:
line_size = 64
element_size = 8
addresses = [i * element_size for i in range(16)]
mapping = {i: address // line_size for i, address in enumerate(addresses)}
assert len({mapping[i] for i in range(8)}) == 1
for index, line in mapping.items(): print(f"contador {index:2} -> línea {line}")
print("separación mínima en elementos:", line_size // element_size)


**Interpretación.** Separar o alinear contadores puede reducir false sharing, pero aumenta memoria y debe medirse.


## Práctica reproducible

1. Estimar bytes transferidos y FLOP de un kernel.
2. Medir una referencia con tamaño que exceda caché cuando la pregunta sea ancho de banda.
3. Registrar afinidad y política NUMA junto con la curva.


## Errores frecuentes

- Usar FLOP/s de pico de otra precisión.
- Confundir misses con prueba automática de false sharing.
- Comparar tamaños que realizan cantidades distintas de trabajo.

## Criterios de aceptación

- Unidades y precisión explícitas.
- Punto Roofline calculado y medición diferenciada.
- Hipótesis de memoria contrastada con al menos un contador o experimento controlado.


## Referencias y material relacionado

- [Planeación: memoria](../../../docs/PLANEACION_CURSO.md)
- [Protocolo](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 01](README.md)
